In [ ]:
import numpy as np 
import pandas as pd 

In [ ]:
dtype_list = {'ind_cco_fin_ult1': 'uint8', 'ind_deme_fin_ult1': 'uint8',
          'ind_aval_fin_ult1': 'uint8', 'ind_valo_fin_ult1': 'uint8',
          'ind_reca_fin_ult1': 'uint8', 'ind_ctju_fin_ult1': 'uint8',
          'ind_cder_fin_ult1': 'uint8', 'ind_plan_fin_ult1': 'uint8',
          'ind_fond_fin_ult1': 'uint8', 'ind_hip_fin_ult1': 'uint8',
          'ind_pres_fin_ult1': 'uint8', 'ind_nomina_ult1': 'Int64', 
          'ind_cno_fin_ult1': 'uint8', 'ind_ctpp_fin_ult1': 'uint8',
          'ind_ahor_fin_ult1': 'uint8', 'ind_dela_fin_ult1': 'uint8',
          'ind_ecue_fin_ult1': 'uint8', 'ind_nom_pens_ult1': 'Int64',
          'ind_recibo_ult1': 'uint8', 'ind_deco_fin_ult1': 'uint8',
          'ind_tjcr_fin_ult1': 'uint8', 'ind_ctop_fin_ult1': 'uint8',
          'ind_viv_fin_ult1': 'uint8', 'ind_ctma_fin_ult1': 'uint8',
         'ncodpers' : 'uint32'}  

name_col = ['ncodpers', 'fecha_dato', 'ind_ahor_fin_ult1','ind_aval_fin_ult1','ind_cco_fin_ult1',
           'ind_cder_fin_ult1','ind_cno_fin_ult1','ind_ctju_fin_ult1',
           'ind_ctma_fin_ult1','ind_ctop_fin_ult1','ind_ctpp_fin_ult1',
           'ind_deco_fin_ult1','ind_deme_fin_ult1','ind_dela_fin_ult1',
           'ind_ecue_fin_ult1','ind_fond_fin_ult1','ind_hip_fin_ult1',
           'ind_plan_fin_ult1','ind_pres_fin_ult1','ind_reca_fin_ult1',
           'ind_tjcr_fin_ult1','ind_valo_fin_ult1','ind_viv_fin_ult1',
           'ind_nomina_ult1','ind_nom_pens_ult1','ind_recibo_ult1']

In [ ]:
def popularity_based(df):
    top_col = {}
    for col in df.columns[1:]:
        top_col[col] = df[col].value_counts()[1]
    
    for k, v in top_col.items():
        top_col[k] = np.around(v / df.shape[0], decimals=4)
        
    return top_col

In [ ]:
df_train1505 = pd.read_csv("data/train_2015_05_28.csv", dtype=dtype_list, usecols=name_col)

In [ ]:
df_train1505.shape

In [ ]:
df_train1505.head()

In [ ]:
df_train1505.drop(['fecha_dato'], axis=1, inplace=True)

In [ ]:
df_train1505 = df_train1505.fillna(0)

In [ ]:
df_ui = df_train1505.copy()

df_ui = df_ui[:50000]
df_ui = df_ui.set_index('ncodpers')
df_ui.shape

In [ ]:
from sklearn.metrics.pairwise import pairwise_distances 

# create the user-item similarity matrix
# removes index names
cosine_sim = 1 - pairwise_distances(df_ui, metric="cosine")

In [ ]:
df_ui.head()

In [ ]:
df_ui.info()

In [ ]:
def useritem(user_id, df, sim_matrix = cosine_sim):
    """
    Function that calculates recommendations for a given user.
    It uses cosine similarity to calculate the most similar users.
    Returns the probability of products for a given user based on similar users.
    Probability range is <0, 1>.
    """
    # computes the index in the user-item similarity matrix for a given user_id
    cos_id = list(df.index).index(user_id) 
    
    # number of similar users
    k = 0
    sim_min = 0.79
    user_sim_k = {}
    
    while k < 20:
        # creates the dictionary {'similar user':'similarity'}
        for user in range(len(df)):
            # 0.99 because I don`t want the same user as user_id
            if sim_min < sim_matrix[cos_id, user] < 0.99:
                user_sim_k[user] = sim_matrix[cos_id, user]
                k+=1
                
        sim_min -= 0.025
        
        # if there are no users with similarity at least 0.65, the recommendation probability will be set to 0 
        if sim_min < 0.65:
            break
            
    # sorted k most similar users
    user_sim_k = dict(sorted(user_sim_k.items(), key=lambda item: item[1], reverse=True))
    user_id_k = list(user_sim_k.keys()) 
    
    # dataframe with k most similar users
    df_user_k = df.iloc[user_id_k]
    df_user_k_T = df_user_k.T
    
    # change the user index to the cosine index
    df_user_k_T.columns = user_id_k
    
    # mean of ownership by k similar users
    ownership = []
    usit = {}
    
    for row_name, row in df_user_k_T.iterrows():
        for indx, own in row.items():
            ownership.append(own) 
        
        usit[row_name] = np.mean(ownership)
        ownership = []
        
    # if there are no users with similarity at least 0.65, the recommendation probability is 0 
    if pd.isna(list(usit.values())[0]) == True:
        usit = {key : 0 for (key, value) in usit.items()}
            
    return usit

In [ ]:
df_ui.head()

In [ ]:
useritem(1061260, df_ui)

In [ ]:
df_mb = df_train1505.copy()
df_mb = df_mb.set_index('ncodpers')

In [ ]:
df_mb = df_mb.fillna("0")

In [ ]:
from collections import defaultdict
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#crie uma lista de modelos para cada produto
models = defaultdict(list)

for c in df_mb.columns:
    if c != 'ncodpers':
        print(c)
        y = df_mb[c].astype('int')
        model = DecisionTreeClassifier(max_depth=10, splitter='random', criterion='entropy', min_samples_leaf=1, min_samples_split=2, random_state=42)
        model.fit(df_mb.drop([c], axis=1), y.values)
        models[c] = model

In [ ]:
x_train = df_mb.drop("ind_ahor_fin_ult1", axis=1)
models["ind_ahor_fin_ult1"].predict_proba(x_train[x_train.index == 1061336])[:, 1]

In [ ]:
def modelbased(user_id, df, models):
    """
    Function that calculates recommendations for a given user.
    It uses machine learning model to calculate the probability of products.
    Probability range is <0, 1>.   
    """
    
    mdbs = {}
    i = 0
    
    for c in df.columns:
        #y_train = df[c].astype('int')
        x_train = df.drop([c], axis=1)
        #model.fit(x_train, y_train)
        
        model = models[c]
        p_train = model.predict_proba(x_train[x_train.index == user_id])[:, 1]
        
        mdbs[c] = p_train[0]
        i += 1
        
    return mdbs

In [ ]:
modelbased(1061336, df_mb, models)

In [ ]:
def hybrid(user_id, df_p, df_u, df_m, f1, f2, f3):
    """
    Function that calculates weighted hybrid recommendations for a given user.
    It uses weights to calculate the probability of products. 
    """
    pb_h = popularity_based(df_p)
    ui_h = useritem(user_id, df_u)
    mb_h =  modelbased(user_id, df_m, models)

    hybrid = {}
    for k, v in pb_h.items():
        hybrid[k] = (v * f1) + (ui_h[k] * f2) #+ (mb_h[k] * f3)
    
    return hybrid

In [ ]:
hybrid_rec_1061608 = hybrid(1061336, df_p = df_train1505, df_u = df_ui, df_m = df_mb, f1 = 0.5, f2 = 0.25, f3 = 0.25)
hybrid_rec_1061608

In [ ]:
def recommendation(user_id, df, hybrid_outcome):
    """
    Function that returns a list of recommendations for a given user.
    """
        
    # products that the user already owns
    user_row = df[df.index == user_id]
    user_products = list(filter(lambda product: user_row[product].to_numpy()[0]==1, user_row))
                
    # removes products that the user already owns
    recom = { key : hybrid_outcome[key] for key in hybrid_outcome if key not in user_products}
    recom_sort = dict(sorted(recom.items(), key=lambda item: item[1], reverse=True))
    
    return list(recom_sort.keys())[:7]

In [ ]:
rec_1061608 = recommendation(1061608, df_mb, hybrid_rec_1061608)

In [ ]:
rec_1061608

In [ ]:
df_test1605 = pd.read_csv("data/train_2016_05_28.csv", dtype=dtype_list, usecols=name_col)

In [ ]:
df_test1605.head()

In [ ]:
df_test1605.drop(['fecha_dato'], axis=1, inplace=True)

In [ ]:
def rec_test(user_id, df1, df2):
    """
    Function that returns a list of test recommendations for a given user.
    """
    recom_test = []
    
    data_before = df1[df1.ncodpers == user_id].values[0]
    data_after = df2[df2.ncodpers == user_id].values[0]
    
    for i, col in enumerate(df1.columns[1:]):    
        if data_before[i+1] == 0 and data_after[i+1] == 1:
            recom_test.append(col)

    return recom_test

In [ ]:
rec_test_1061608 = rec_test(1061608, df_train1505, df_test1605)

In [ ]:
rec_test_1061608

In [ ]:
# referring to https://github.com/benhamner/Metrics/blob/master/Python/ml_metrics/average_precision.py

def apk(actual, predicted, k=7):
    """
    Computes the average precision at k.
    This function computes the average prescision at k between two lists of
    items.
    Parameters
    ----------
    actual : list
             A list of elements that are to be predicted (order doesn't matter)
    predicted : list
                A list of predicted elements (order does matter)
    k : int, optional
        The maximum number of predicted elements
    Returns
    -------
    score : double
            The average precision at k over the input lists
    """
    if len(predicted)>k:
        predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i,p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i+1.0)

    if not actual:
        return 0.0

    return score / min(len(actual), k)

def mapk(actual, predicted, k=7):
    """
    Computes the mean average precision at k.
    This function computes the mean average prescision at k between two lists
    of lists of items.
    Parameters
    ----------
    actual : list
             A list of lists of elements that are to be predicted 
             (order doesn't matter in the lists)
    predicted : list
                A list of lists of predicted elements
                (order matters in the lists)
    k : int, optional
        The maximum number of predicted elements
    Returns
    -------
    score : double
            The mean average precision at k over the input lists
    """
    return np.mean([apk(a,p,k) for a,p in zip(actual, predicted)])

In [ ]:
def evaluation(user_id, f1, f2, f3):
    """
    Function that returns the average precision metric for a given user.
    """
    y_real = rec_test(user_id, df_train1505, df_test1605)
    y_pred = recommendation(user_id, df_mb,
                            hybrid_outcome = hybrid(user_id, df_u = df_ui, df_m = df_mb,
                                                    df_p = df_train1505,
                                                    f1 = f1, f2 = f2, f3 = f3))
    
    return y_real, y_pred, apk(y_real, y_pred)

In [ ]:
df_train1505.head()

In [ ]:
df_final = df_train1505.merge(df_test1605, on="ncodpers", how="inner")[["ncodpers"]]

In [ ]:
df_final = df_final.merge(df_ui, on="ncodpers", how="inner")
sample = df_final['ncodpers']

In [ ]:
sample.shape

In [ ]:
from tqdm import tqdm

In [ ]:
score_list  = []
y_pred_list = []
y_real_list = []

for cod in tqdm(sample):
    y_real, y_pred, score = evaluation(cod, f1=0.8, f2=0.5, f3=0.2)
    score_list.append(score)
    y_pred_list.append(y_pred)
    y_real_list.append(y_real)

In [ ]:
y_pred_list

In [ ]:
df_final['score'] = score_list
df_final['y_pred'] = y_pred_list
df_final['y_real'] = y_real_list

In [ ]:
df_final = df_final[['ncodpers', 'recom', 'score', 'y_pred', 'y_real']]

In [ ]:
df_final = df_final[df_final["recom"] > 0]

In [ ]:
df_final

In [ ]:
np.mean(score_list)

In [ ]:
df_final.to_csv("data/hybrid_recomendation.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_final = pd.read_csv("data/hybrid_recomendation.csv")

In [ ]:
df_final.head()

In [ ]:
import ast

#transform the column of list into a list type
df_final["y_pred"] = df_final["y_pred"].apply(ast.literal_eval)
df_final["y_real"] = df_final["y_real"].apply(ast.literal_eval)

In [ ]:
df_final.head()

In [ ]:
rename_products = {
    "ind_ahor_fin_ult1": "saving_account",
    "ind_aval_fin_ult1": "guarantees",
    "ind_cco_fin_ult1": "current_accounts",
    "ind_cder_fin_ult1": "derivada_account",
    "ind_cno_fin_ult1": "payroll_account",
    "ind_ctju_fin_ult1": "junior_account",
    "ind_ctma_fin_ult1": "mas_particular_account",
    "ind_ctop_fin_ult1": "particular_account",
    "ind_ctpp_fin_ult1": "particular_plus_account",
    "ind_deco_fin_ult1": "short_term_deposits",
    "ind_deme_fin_ult1": "medium_term_deposits",
    "ind_dela_fin_ult1": "long_term_deposits",
    "ind_ecue_fin_ult1": "e_account",
    "ind_fond_fin_ult1": "funds",
    "ind_hip_fin_ult1": "mortgage",
    "ind_plan_fin_ult1": "pensions",
    "ind_pres_fin_ult1": "loans",
    "ind_reca_fin_ult1": "taxes",
    "ind_tjcr_fin_ult1": "credit_card",
    "ind_valo_fin_ult1": "securities",
    "ind_viv_fin_ult1": "home_account",
    "ind_nomina_ult1": "payroll",
    "ind_nom_pens_ult1": "pensions",
    "ind_recibo_ult1": "direct_debit",
}

In [ ]:
#rename y_pred and y_real columns
df_final["y_pred"] = df_final["y_pred"].apply(lambda x: [rename_products[i] for i in x])
df_final["y_real"] = df_final["y_real"].apply(lambda x: [rename_products[i] for i in x])

In [ ]:
df_final.head()

In [ ]:
sns.set_style("darkgrid")

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df_final['score'], bins=10, kde=True, color="g")
plt.xlabel("MAP@7")
plt.ylabel("Frequência")
#plt.title("Media de MAP@7 das recomendações")
plt.show()

In [ ]:
df_final['quantile'] = pd.cut(df_final['score'], bins=[-np.inf, 0, 0.25, 0.5, 0.75, 1], labels=['Sem Acerto', '(0% - 25%]', '(25% - 50%]', '(50% - 75%]', '(75% - 100%]'])

In [ ]:
#Separe de 20 em 20% para plotar um grafico de barras baseado na variavel mapk_score
plt.figure(figsize=(10, 5))
sns.countplot(x="quantile", data=df_final, color="g")
plt.xlabel("MAP@7")
plt.ylabel("Frequência")
#plt.title("Porcentagem MAP@7 das recomendações")
plt.show()

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import confusion_matrix

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

compras_reais = df_final['y_real']
recomendacoes = df_final['y_pred']

df = pd.DataFrame({'Recomendacoes': recomendacoes, 'Compras': compras_reais})

# Inicialize listas para armazenar as métricas calculadas
precision_list = []
accuracy_list = []
recall_list = []
f1_score_list = []
specificity_list = []

for i in range(1, 8):
    # Obtenha as listas de recomendações e compras reais
    compras_reais = df['Compras'].tolist()
    recomendacoes = df['Recomendacoes'].apply(lambda x: x[:i]).tolist()

    # Inicialize as listas para armazenar as predições binárias (0 ou 1)
    y_true = []
    y_pred = []

    # Crie as listas de predições binárias com base nas recomendações e compras reais
    for compras, recs in zip(compras_reais, recomendacoes):
        # Transforme as listas em conjuntos para facilitar as operações de união e interseção
        compras_set = set(compras)
        recs_set = set(recs)

        # Se o produto recomendado estiver nas compras reais, a predição é verdadeira (1),
        # caso contrário, a predição é falsa (0)
        for prod in recs_set:
            y_true.append(1 if prod in compras_set else 0)
            y_pred.append(1)

        for prod in compras_set:
            if prod not in recs_set:
                y_true.append(1)
                y_pred.append(0)

    # Calcule as métricas para este valor de i e adicione-os às listas de resultados
    precision_list.append(precision_score(y_true, y_pred))
    accuracy_list.append(accuracy_score(y_true, y_pred))
    recall_list.append(recall_score(y_true, y_pred))
    f1_score_list.append(f1_score(y_true, y_pred))

# Agora você pode usar as listas de métricas para análise posterior
print("Accuracy:", accuracy_list)
print("Precision:", precision_list)
print("Recall:", recall_list)
print("F1 Score:", f1_score_list)


In [ ]:
for i in range(1, 8):

    compras_reais = df_final['y_real']
    recomendacoes = df_final['y_pred'].apply(lambda x: x[:i])

    df = pd.DataFrame({'Recomendacoes': recomendacoes, 'Compras': compras_reais})

    # Inicializando as variáveis para as métricas
    total_recomendacoes = 0
    acertos = 0
    total_compras = 0
    recomendacoes_corretas = 0
    falsos_positivos = 0

    for idx, row in df.iterrows():
        recomendacoes = set(row['Recomendacoes'])
        compras = set(row['Compras'])

        total_recomendacoes += len(recomendacoes)
        total_compras += len(compras)

        for recom in recomendacoes:
            if recom in compras:
                acertos += 1
                recomendacoes_corretas += 1
            else:
                falsos_positivos += 1

    # Calculando as métricas
    acuracia = acertos / total_recomendacoes
    recall = recomendacoes_corretas / total_compras
    precisao = recomendacoes_corretas / total_recomendacoes
    especificidade = (total_compras - recomendacoes_corretas) / (total_compras - total_recomendacoes)
    f1_score = 2 * (precisao * recall) / (precisao + recall)

    print("\nMétricas para o top", i)
    print("Acurácia:", acuracia)
    print("Recall:", recall)
    print("Precisão:", precisao)
    print("Especificidade:", especificidade)
    print("F1-Score:", f1_score)

In [ ]:
compras_reais = df_final['y_real']
recomendacoes = df_final['y_pred'].apply(lambda x: x[:7])

df = pd.DataFrame({'Recomendacoes': recomendacoes, 'Compras': compras_reais})

In [ ]:
# Função para calcular a matriz de confusão
def calculate_confusion_matrix(df):
    classes = sorted(list(set([item for sublist in df['Recomendacoes'] for item in sublist])))
    n_classes = len(classes)

    confusion = np.zeros((n_classes, n_classes), dtype=int)

    for _, row in df.iterrows():
        recomendacoes = set(row['Recomendacoes'])
        compras = set(row['Compras'])

        for recom_idx, recom_class in enumerate(classes):
            for compra_idx, compra_class in enumerate(classes):
                if recom_class in recomendacoes and compra_class in compras:
                    confusion[recom_idx][compra_idx] += 1

    return confusion, classes

# Calculando a matriz de confusão e as classes
confusion_matrix, classes = calculate_confusion_matrix(df)

print("Matriz de Confusão:")
plt.figure(figsize=(14, 6))
sns.heatmap(confusion_matrix, annot=True, fmt="d", xticklabels=classes, yticklabels=classes)

plt.show()
#print("\nClasses:", classes)